# Level 2 — Feature Engineering with DuckDB

## Objective

This notebook loads the cleaned dataset into DuckDB and creates engineered features that support downstream business analysis.

### Load Dataset into DuckDB

Load the cleaned dataset into DuckDB to begin feature engineering and analytical processing.

In [12]:
from pathlib import Path
import duckdb

# Path to the cleaned dataset
data_path = Path("../data/superstore_utf8.csv")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW superstore AS
SELECT *
FROM read_csv_auto('{data_path.as_posix()}', header=True)
""")

con.execute("""
SELECT *
FROM superstore
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Feature Engineering

A DuckDB view named `superstore_features` was created to preserve the original dataset while adding analytical features. New columns include fulfillment time, profit margin, order year and month, and customer lifetime sales calculated using a window function. These engineered features support customer segmentation, profitability analysis, and operational performance reporting.

In [13]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    date_diff('day', order_date, ship_date) AS fulfillment_days,
    profit / sales AS profit_margin,
    year(order_date) AS order_year,
    month(order_date) AS order_month,
    SUM(sales) OVER (
        PARTITION BY customer_id
    ) AS customer_lifetime_sales
FROM superstore
""")

## Quick Inspection of Newly Created Features
The newly created features were queried from the `superstore_features` view and inspected using a sample of 10 records. This validation step ensured that the feature engineering process produced the expected values before proceeding with further analysis.

In [14]:
con.execute("""
SELECT
    customer_name,
    fulfillment_days,
    profit_margin,
    order_year,
    order_month,
    customer_lifetime_sales
FROM superstore_features
LIMIT 10
""").df()

,customer_name,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales
0,Art Foster,4,0.237500,2014,5,861.565
1,Art Foster,2,-0.225000,2016,11,861.565
2,Art Foster,5,-0.733333,2014,9,861.565
3,Art Foster,2,-0.062500,2016,11,861.565
4,Art Foster,4,-0.620000,2014,5,861.565
5,Art Foster,4,0.280000,2015,10,861.565
6,Art Foster,4,0.270000,2015,10,861.565
7,Arthur Gainer,4,0.175000,2014,7,4510.797
8,Arthur Gainer,4,0.260000,2016,12,4510.797
9,Arthur Gainer,1,0.290000,2016,5,4510.797


## Fulfillment Days Analysis

The distribution of the engineered `fulfillment_days` feature was examined to verify that the calculated values were reasonable. Most orders were fulfilled within **4–5 days**, with relatively few orders requiring **0–1 days** or the maximum of **7 days**, indicating a realistic distribution suitable for downstream operational analysis.

In [15]:
con.execute('''
SELECT
    fulfillment_days,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY fulfillment_days
ORDER BY fulfillment_days;
''').df()            

,fulfillment_days,orders
0,0,519
1,1,369
2,2,1334
3,3,1005
4,4,2774
5,5,2169
6,6,1203
7,7,621


## Profit Margin Analysis

Analysis of the engineered `profit_margin` feature showed an average profit margin of **12.03%**, indicating that the company earned approximately 12 cents of profit for every dollar of sales. Profit margins ranged from **-275%** to **50%**, highlighting that while some transactions were highly profitable, others resulted in substantial losses.

In [16]:
con.execute('''
SELECT
    ROUND(AVG(profit_margin),4) AS avg_profit_margin,
    MIN(profit_margin) AS min_margin,
    MEDIAN(profit_margin) AS median_margin,
    MAX(profit_margin) AS max_margin
FROM superstore_features;
''').df()

,avg_profit_margin,min_margin,median_margin,max_margin
0,0.1203,-2.75,0.27,0.5


## Lowest Profit Margin Products

The products with the lowest profit margins were identified to better understand the transactions contributing to overall losses. Several products exhibited profit margins below **-270%**, indicating that the losses incurred on these sales substantially exceeded the revenue generated, making them potential candidates for further pricing or discount analysis.

In [17]:
con.execute(''' 
SELECT
    product_name,
    sales,
    profit,
    profit_margin
FROM superstore_features
ORDER BY profit_margin
LIMIT 10
''').df()

,product_name,sales,profit,profit_margin
0,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.624,-4.4660,-2.75
1,Kensington 6 Outlet SmartSocket Surge Protector,24.588,-67.6170,-2.75
2,Hoover Portapower Portable Vacuum,2.688,-7.3920,-2.75
3,Hoover Shoulder Vac Commercial Portable Vacuum,143.128,-393.6020,-2.75
4,Hoover Commercial Lightweight Upright Vacuum w...,93.032,-251.1864,-2.70
5,Belkin 7-Outlet SurgeMaster Home Series,5.588,-15.0876,-2.70
6,Fellowes 8 Outlet Superior Workstation Surge P...,33.620,-90.7740,-2.70
7,Hoover Commercial Lightweight Upright Vacuum,1.392,-3.7584,-2.70
8,Tripp Lite Isotel 8 Ultra 8 Outlet Metal Surge,70.970,-191.6190,-2.70
9,Belkin 6 Outlet Metallic Surge Strip,4.356,-11.7612,-2.70


## Customer Lifetime Sales Analysis

The `customer_lifetime_sales` feature was used to identify the highest-value customers in the dataset. Sean Miller generated over **$25,000** in lifetime sales, while several other customers exceeded **$12,000** in total purchases, indicating that a relatively small group of customers contributed substantially to overall revenue.

In [18]:
con.execute('''
SELECT DISTINCT
    customer_name,
    customer_lifetime_sales
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,customer_lifetime_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


In [19]:
con.execute('''
SELECT
    quantile_cont(customer_lifetime_sales, 0.25) AS q1,
    quantile_cont(customer_lifetime_sales, 0.50) AS median,
    quantile_cont(customer_lifetime_sales, 0.75) AS q3,
    quantile_cont(customer_lifetime_sales, 0.9) AS top_10,
FROM (
    SELECT DISTINCT
        customer_id,
        customer_lifetime_sales
    FROM superstore_features
);
''').df()

,q1,median,q3,top_10
0,1146.05,2256.394,3785.276,6038.48


In [20]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    CASE
        WHEN customer_lifetime_sales >= 6038.48 THEN 'Big Fish'
        WHEN customer_lifetime_sales >= 3785.276 THEN 'Premium'
        WHEN customer_lifetime_sales >= 2256.394 THEN 'High Value'
        ELSE 'Standard'
    END AS customer_tier
FROM (
    SELECT
        *,
        date_diff('day', order_date, ship_date) AS fulfillment_days,
        profit / sales AS profit_margin,
        year(order_date) AS order_year,
        month(order_date) AS order_month,
        SUM(sales) OVER (
            PARTITION BY customer_id
        ) AS customer_lifetime_sales
    FROM superstore
);
""")

## Customer Segmentation

Customers were segmented according to their lifetime sales using percentile-based thresholds. Customers in the top 10% of lifetime spending were classified as "Big Fish," while the remaining customers were categorized as Premium, High Value, or Standard based on the 75th and 50th percentile cutoffs. This feature provides a business-oriented method for identifying and analyzing high-value customers.

In [21]:
con.execute('''
SELECT DISTINCT
        customer_name,
        customer_lifetime_sales,
        customer_tier
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,customer_lifetime_sales,customer_tier
0,Sean Miller,25043.050,Big Fish
1,Tamara Chand,19052.218,Big Fish
2,Raymond Buch,15117.339,Big Fish
3,Tom Ashbrook,14595.620,Big Fish
4,Adrian Barton,14473.571,Big Fish
5,Ken Lonsdale,14175.229,Big Fish
6,Sanjit Chand,14142.334,Big Fish
7,Hunter Lopez,12873.298,Big Fish
8,Sanjit Engle,12209.438,Big Fish
9,Christopher Conant,12129.072,Big Fish


In [22]:
con.execute('''
SELECT
    order_year,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
''').df()

,order_year,orders
0,2014,1993
1,2015,2102
2,2016,2587
3,2017,3312
